### Read from the Source System (CSV)

In [0]:
# Import the Bronze configuration (defines all CSV paths)
from scripts.bronze_config import BRONZE_CONFIG

# Display what we're about to load
print("=" * 70)
print("BRONZE LAYER: Loading all source tables from CSVs")
print("=" * 70)
print(f"\nFound {len(BRONZE_CONFIG)} tables to load:\n")
for table_name, path in BRONZE_CONFIG.items():
    print(f"  • {table_name}")

### Load All CSVs to Bronze Delta Tables

In [0]:
# Loop through each CSV file and load it to Bronze
for table_name, csv_path in BRONZE_CONFIG.items():
    print(f"\n{'='*70}")
    print(f"Loading: {table_name}")
    print(f"{'='*70}")
    
    # Step 1: Read CSV into a DataFrame
    # - header=true: First row contains column names
    # - inferSchema=true: Spark detects data types (int, string, date, etc.)
    df = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .csv(csv_path)
    
    # Step 2: Count rows and show schema
    row_count = df.count()
    col_count = len(df.columns)
    print(f"  Rows: {row_count:,} | Columns: {col_count}")
    
    # Step 3: Write to Bronze schema as Delta table
    # - mode("overwrite"): If table exists, replace it (clean slate each run)
    # - format("delta"): Use Delta Lake format (reliable, auditable, ACID-compliant)
    # - saveAsTable(): Create a permanent table in the catalog
    full_table_name = f"`databricks-medallion-lakehouse`.bronze.{table_name}"
    
    df.write \
        .mode("overwrite") \
        .format("delta") \
        .saveAsTable(full_table_name)
    
    print(f"  ✅ Written to: {full_table_name}")

print(f"\n{'='*70}")
print("✅ BRONZE LAYER COMPLETE: All 6 tables loaded")
print(f"{'='*70}")

### Verify Tables Exist 

In [0]:
# Query the Bronze schema to verify all tables were created
bronze_tables = spark.sql("""
   SELECT table_name 
    FROM `databricks-medallion-lakehouse`.information_schema.tables 
    WHERE table_schema = 'bronze'
    ORDER BY table_name
""")

print("✅ Tables in Bronze schema:")
bronze_tables.show()